# Challenge 08: Script May Crash Halfway

**PPT problem:** Script may crash halfway  
**Technique:** Checkpoint / resume

A checkpoint is a small piece of saved progress that lets a long-running job continue after a crash.

Without a checkpoint, a job has only two choices after failure:

1. Start again from the beginning, which wastes time and may duplicate records.
2. Stop permanently, which leaves the dataset incomplete.

With a checkpoint, the job can say: **I already finished up to here, so I will continue from the next unit of work.**


## Checkpoint Mental Model

Think of an API ingestion job as a sequence of small units of work.

Common units of work:

| API pattern | Checkpoint can store |
|---|---|
| Page-based API | `last_completed_page` |
| Cursor-based API | `next_cursor` or `last_successful_cursor` |
| Time-window API | `last_successful_timestamp` |
| File or stream processing | byte offset, line number, or event offset |

The exact field is different, but the idea is the same: save enough state to know where to continue.


## What Should Be Saved?

A robust checkpoint design usually separates two things:

| File/table | Purpose | Example |
|---|---|---|
| Data output | Stores records already collected | `checkpoint_records.jsonl` |
| Checkpoint state | Stores progress marker | `checkpoint.json` |

For this exercise, a simple checkpoint can be:

```json
{
  "last_completed_page": 3
}
```

After a crash, the next run starts from page `4`.


## Correct Write Order

A common safe order is:

1. Request one page from the API.
2. Validate and normalize that page.
3. Write the records for that page to durable storage.
4. Then update the checkpoint to say the page is complete.

Do not update the checkpoint before saving the data. If the program crashes after updating the checkpoint but before writing records, the next run may skip data.


## Duplicates Still Matter

Checkpointing reduces repeated work, but it does not remove the need for idempotency.

A robust ingestion job should still handle reruns safely:

- Use a stable unique key such as `record_id`.
- Drop duplicates before final output, or insert into a database with a unique constraint.
- Make rerunning the job produce the same final table.

In interviews, this is often described as making the pipeline **idempotent**.


In [ ]:
# Design sketch only. This is not the completed repair.
#
# checkpoint = read_checkpoint()
# page = checkpoint["last_completed_page"] + 1
#
# while True:
#     payload = request_page(page)
#     records = normalize(payload["data"])
#     append_records(records)          # save data first
#     write_checkpoint(page)           # then mark page complete
#
#     if payload["next_page"] is None:
#         break
#     page = payload["next_page"]


## Setup

Make sure the mock serving API is running:

```bash
cd day_2/day_2_1_API
python -m uvicorn mock_serving_api:app --reload --port 8011
```

If you use a different local port, update `BASE_URL` in the code cell.


## Run the Broken Program First

Do not fix the code before running it. Run it once and observe the failure signal.

Look for:

- Which page completed before the crash?
- Was any output saved before the crash?
- What state would be enough to resume?



In [ ]:
"""
Challenge 08: Checkpoint / resume.

This long paginated job crashes after page 3. Because the script only writes
the output at the very end, all progress is lost.
"""

import pandas as pd
import requests


BASE_URL = "http://127.0.0.1:8011"
CLIENT_ID = "group_08_checkpoint"


def main() -> None:
    page = 1
    all_records = []

    while True:
        response = requests.get(
            f"{BASE_URL}/debug-lab/08-checkpoint/taxi/records",
            params={
                "page": page,
                "page_size": 20,
                "client_id": CLIENT_ID,
            },
            timeout=10,
        )
        response.raise_for_status()
        payload = response.json()
        all_records.extend(payload["data"])
        print(f"Finished page {page}")

        if page == 3:
            raise RuntimeError("Simulated crash after page 3. Progress was not saved.")

        if payload["next_page"] is None:
            break
        page = payload["next_page"]

    dataframe = pd.DataFrame(all_records)
    dataframe.to_csv("checkpoint_output.csv", index=False)
    print("Rows written:", len(dataframe))


if __name__ == "__main__":
    main()


## Diagnose

Write short answers before changing the code.

1. Which pages finished before the simulated crash?
2. Did the script save any records before crashing?
3. If you rerun the script unchanged, what work will repeat?
4. What should the checkpoint store for this page-based API?
5. What is the safe write order: checkpoint first or records first?
6. How will you avoid duplicate records after reruns?


In [ ]:
# Notes
# 1.
# 2.
# 3.
# 4.
# 5.
# 6.


## Where to Look for the Fix

Use the API docs, the observed failure, and the clues below to decide what to change.

Primary place to check: `http://127.0.0.1:8011/docs`, then find `/debug-lab/08-checkpoint/taxi/records`.

Use these clues:

1. Use the checkpoint mental model at the top of this notebook.
2. The key idea is save records first, then mark the page complete.
3. The repaired client should resume from the next unfinished page and avoid duplicate records.

After that, use the success condition below to check whether your repaired code is good enough.


## Repair Workspace

Copy the broken code here and edit it until it works.

Success condition: the notebook should save records after each completed page, update a checkpoint, simulate a crash, and then resume from the next page instead of starting from page 1.

Suggested local files:

```text
checkpoint.json
checkpoint_records.jsonl
```


In [ ]:
# Paste the broken code here, then repair it.
# Start by checking /docs for the endpoint contract and rereading the error output above.
# Stop when the success condition in the previous markdown cell is satisfied.
